# Intensidade da Ilha de Calor Urbana: Dossel (Ar) e Superfície (Satélite)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/03_urban_heat_island.pt.ipynb)

**Objetivo de aprendizagem.** Este notebook mostra como quantificar a intensidade da Ilha de Calor Urbana (UHI) de duas formas complementares: (1) a clássica **UHI de dossel** (canopy-layer), calculada a partir de dados de temperatura do ar de estações estratificadas por LCZ com `lcz_uhi_intensity`, e (2) a **UHI de Superfície (SUHI)**, calculada a partir da Temperatura da Superfície Terrestre (LST) por satélite com `lcz_uhi_surface`, que oferece sete métodos diagnósticos distintos. Ao final, você será capaz de calcular ambas, explicar por que não são intercambiáveis, e escolher o diagnóstico de SUHI certo para cada pergunta.

> **Nota de renderização:** todas as figuras deste caderno são geradas por padrão com o renderizador estático nativo do LCZ4py (matplotlib, exportação em 300 dpi PNG/PDF/SVG/TIFF, sem precisar de Plotly/Kaleido). Passe `renderer="plotly"` para uma função de plotagem para obter a figura interativa do Plotly.

## Por que duas "UHI" diferentes?

A Ilha de Calor Urbana não é uma única grandeza física — é uma família de fenômenos relacionados, porém distintos, e confundi-los é um dos erros mais comuns em estudos de clima urbano (Voogt & Oke, 2003, *"Thermal remote sensing of urban climates"*, Remote Sensing of Environment).

- A **UHI de dossel (canopy-layer, CUHI)** é a diferença de temperatura medida por termômetros à altura do abrigo meteorológico (~2 m), dentro da camada de dossel urbano — o espaço entre as superfícies dos edifícios e o topo do dossel vegetal/telhados. É o que um pedestre sente, o que uma estação meteorológica registra, e o que a maioria dos estudos epidemiológicos de calor-saúde utiliza. Tem um ciclo diurno marcante: normalmente atinge o pico algumas horas após o pôr do sol, quando as áreas rurais resfriam radiativamente mais rápido do que as urbanas, que liberam o calor armazenado durante o dia (efeito de "armazenamento de calor urbano" de materiais de alta admitância térmica).
- A **UHI de Superfície (SUHI)** é a diferença na *temperatura da superfície terrestre* (temperatura da "pele" de telhados, pavimentos, solo, topo do dossel vegetal), obtida por sensores infravermelhos termais em satélites. Normalmente atinge o pico perto do meio-dia solar, quando o sol aquece superfícies impermeáveis muito mais do que o ar acima delas, podendo ser várias vezes maior em magnitude do que a UHI de dossel no mesmo local e horário. SUHI e CUHI podem até apresentar cronologia diurna *oposta* e, em alguns biomas, sinal oposto em certos horários.

Como essas duas UHIs medem processos físicos diferentes em alturas, horários e resoluções espaciais distintos, este notebook mantém-nas em seções separadas e nunca faz média de uma sobre a outra.

### Por que classificação baseada em LCZ, e não "centro da cidade vs. subúrbio"

Uma fraqueza recorrente na literatura mais antiga de UHI é definir "urbano" e "rural" por um raio arbitrário a partir do centro da cidade, ou por um rótulo grosseiro de uso do solo. O esquema de Zonas Climáticas Locais (Stewart & Oke, 2012, *"Local Climate Zones for Urban Temperature Studies"*, BAMS) classifica cada estação ou pixel pela sua morfologia real de entorno — altura e densidade dos edifícios, fração de superfície permeável/impermeável, fator de visão do céu — independentemente da distância ao centro. Uma estação a 15 km do centro em um bairro denso de LCZ 2 (compacto médio-alto) é legitimamente "urbana" para fins de UHI, enquanto um parque a duas quadras do centro (LCZ 9, esparsamente construído, ou LCZ B/D de cobertura natural) é legitimamente "rural". Tanto `lcz_uhi_intensity` quanto `lcz_uhi_surface` neste pacote adotam por padrão essa convenção: as classes LCZ 1–10 (tipos construídos) são urbanas, as classes LCZ 11–16 (coberturas naturais, excluindo água) são a referência rural. Isso gera uma definição de UHI fisicamente fundamentada, reprodutível entre cidades, e não sensível ao formato particular de uma cidade ou a onde se traça um limite administrativo. A metodologia deste pacote segue o arcabouço do LCZ4r descrito em Anjos, M. et al. (2025), *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1 — Configuração: mapa LCZ e dados de estações de Berlim

Como no restante da série `local/`, usamos Berlim: o mapa LCZ (produto global, Demuzere et al. 2022) via `lcz_get_map`, e `lcz_data_berlin.csv` — temperatura do ar horária de 23 estações em Berlim e arredores (2019–2020).

In [ ]:
import pandas as pd
from IPython.display import display
from LCZ4py import lcz_get_map, lcz_uhi_intensity, lcz_get_lst, lcz_uhi_surface

map_path = lcz_get_map(city="Berlin", isave_map=False)
print("map_path:", map_path)

df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
print(f"{len(df):,} linhas -- {df['station'].nunique()} estacoes -- {df['date'].min()} a {df['date'].max()}")
df.head(3)

## 2 — UHI de dossel a partir de estações: `lcz_uhi_intensity`

`lcz_uhi_intensity(x, data_frame, var, station_id, ...)` calcula a intensidade da UHI como **média(T em estações urbanas) − média(T em estações rurais)**, agregada por `time_freq` ("hour"/"day"/"month"). Com `method="LCZ"` (padrão), a atribuição estação-urbana/rural é automática: cada estação é classificada pelo valor do raster LCZ em suas coordenadas, usando `URBAN_CLASSES = LCZ 1–10` e `RURAL_CLASSES = LCZ 11–16` (constantes do módulo `lcz_uhi_intensity.py`) — a mesma divisão LCZ 1–10 / 11–16 motivada acima, com a LCZ 17 (água) excluída de ambos os grupos. `method="manual"` permite fornecer listas explícitas de IDs de estação `T_urban=[...]`/`T_rural=[...]` para uma divisão personalizada. Use `group=True` para também plotar as séries médias urbana e rural em um eixo secundário junto com a linha de UHI (`group=False`, padrão, plota apenas a UHI). `iplot=False` retorna a tabela larga subjacente (`date, urban, rural, uhi`) em vez de uma figura — útil para análises adicionais.

In [ ]:
fig_uhi = lcz_uhi_intensity(
    map_path, data_frame=df, var="airT", station_id="station",
    time_freq="day", method="LCZ", group=True, iplot=True, lang="pt",
)
display(fig_uhi)

**Interpretando o gráfico.** As linhas sólidas urbana/rural (eixo esquerdo) mostram a temperatura média diária de cada grupo; a linha pontilhada de UHI (eixo direito) é sua diferença. Uma UHI positiva significa que as estações urbanas estão, em média, mais quentes que as rurais naquele dia — o sinal esperado na maioria das cidades na maior parte do tempo. Observe: a *magnitude* (tipicamente 1–3 °C para uma cidade europeia de médio porte como Berlim em resolução diária), e se a UHI acompanha o tempo sinótico (ex.: cai em períodos ventosos/nublados/chuvosos, que reduzem a UHI ao intensificar a mistura, e sobe em condições calmas, claras e de alta pressão que favorecem forte diferencial de resfriamento noturno).

Também é possível obter a tabela numérica bruta em vez de uma figura, o que é conveniente para calcular estatísticas resumo (ex.: a UHI média em todo o período):

In [4]:
uhi_df = lcz_uhi_intensity(
    map_path, data_frame=df, var="airT", station_id="station",
    time_freq="day", method="LCZ", iplot=False, lang="pt",
)
print(f"UHI media diaria: {uhi_df['uhi'].mean():.2f} C  (n={len(uhi_df)} dias)")
uhi_df.head()

UHI media diaria: 0.38 C  (n=700 dias)


date,rural,urban,uhi
datetime[μs],f64,f64,f64
2019-01-01 00:00:00,6.535379,6.598354,0.06
2019-01-02 00:00:00,1.74721,1.954483,0.21
2019-01-03 00:00:00,-0.795958,-0.362402,0.43
2019-01-04 00:00:00,1.525935,1.820119,0.29
2019-01-05 00:00:00,5.862959,5.982546,0.12


## 3 — Obtendo uma pilha de Temperatura da Superfície Terrestre (LST): `lcz_get_lst`

Antes de calcular a UHI de *superfície* precisamos de dados de LST por satélite na mesma grade do mapa LCZ. `lcz_get_lst(x, source, start_date, end_date, ...)` baixa uma pilha diária de LST (uma banda por data) recortada à área do mapa LCZ — veja `notebooks/general/04_remote_sensing_lst_pc` para o passeio completo pelas opções desta função (GOES vs. Sentinel-3 vs. "both"). Aqui só precisamos de uma pilha curta para alimentar `lcz_uhi_surface`. Berlim está fora do domínio CONUS do GOES, então usamos `source="sentinel3"` (global, ~1 km, sem chave via Microsoft Planetary Computer), em uma janela curta de meio do verão (julho, quando o contraste térmico urbano-rural costuma ser mais evidente) dentro da cobertura 2019–2020 do CSV.

In [5]:
lst = lcz_get_lst(
    map_path, source="sentinel3",
    start_date="2019-07-01", end_date="2019-07-06",
    units="C", cache=True,
)
print("datas:", lst.dates)
print("formato do array (dias, H, W):", lst.array.shape)

06:52:16 - LCZ4py.general.lcz_get_lst - INFO - Using cached LST stack: /Users/co2map/.lcz4r_cache/lst_stack_9e7fccd3f1964cc7.tif


datas: ['2019-07-01', '2019-07-02', '2019-07-03', '2019-07-04', '2019-07-05', '2019-07-06']
formato do array (dias, H, W): (6, 377, 750)


## 4 — UHI de Superfície: `lcz_uhi_surface`

`lcz_uhi_surface(lcz_map, lst_x, method=..., ...)` calcula a SUHI a partir da pilha de LST, estratificada por LCZ, com **sete** métodos diagnósticos distintos (`method="all"` executa todos e retorna um dict indexado pelo nome do método). Demonstramos quatro deles abaixo — `urban_rural`, `lcz`, `utfvi` e `hotspot` — cada um respondendo a uma pergunta diferente sobre os mesmos dados de LST subjacentes.

### 4a — `method="urban_rural"`: a série temporal clássica

Por data, `média(LST | LCZ urbana 1–10) − média(LST | LCZ rural 11–16)` — o análogo direto por satélite do que calculamos para a temperatura do ar acima. É o primeiro diagnóstico natural: existe contraste térmico de superfície, e como sua magnitude se compara à UHI de dossel da §2?

In [6]:
suhi_ur = lcz_uhi_surface(map_path, lst, method="urban_rural", lang="pt")
suhi_ur.df

date,urban,rural,suhi,n_urban,n_rural
str,f64,f64,f64,i64,i64
"""2019-07-01""",295.485077,294.930542,0.554535,95024,44778
"""2019-07-02""",290.305725,289.82785,0.477875,95024,44778
"""2019-07-03""",281.641388,280.843323,0.798065,95024,44778
"""2019-07-04""",280.573608,277.690552,2.883057,95024,44778
"""2019-07-05""",279.167969,279.153992,0.013977,95024,44778
"""2019-07-06""",255.281616,253.252533,2.029083,95024,44778


In [ ]:
display(suhi_ur.fig)

**Interpretando.** Compare a magnitude com a UHI de temperatura do ar calculada na §2 — a SUHI costuma ser vários graus maior, pois superfícies expostas/impermeáveis aquecem muito mais sob sol direto do que o ar alguns metros acima delas. `n_urban`/`n_rural` na tabela indicam quantos pixels válidos (sem nuvem/sem dado) entraram na média de cada data — trate com cautela datas com `n_rural` muito baixo.

### 4b — `method="lcz"`: ΔT por classe contra uma classe de referência

Em vez de agrupar todos os pixels urbanos/rurais em dois grupos, este método calcula a LST média de cada uma das 17 classes LCZ menos a classe de referência (`reference_lcz=14`, "Vegetação rasteira" — convenção WUDAPT/Chakraborty & Lee 2019), com média entre as datas. Responde: *qual morfologia urbana específica é a mais quente*, não apenas "urbano vs. rural" como um todo.

In [8]:
suhi_lcz = lcz_uhi_surface(map_path, lst, method="lcz", reference_lcz=14, lang="pt")
suhi_lcz.df.sort("delta_t", descending=True)

date,lcz,mean_lst,delta_t,count
str,i16,f64,f64,i64
"""2019-07-04""",1,283.190674,4.403931,549
"""2019-07-04""",2,282.985474,4.19873,4583
"""2019-07-04""",10,282.748535,3.961792,227
"""2019-07-02""",17,292.799683,3.872559,6352
"""2019-07-04""",4,282.475983,3.68924,3513
…,…,…,…,…
"""2019-07-04""",11,277.184814,-1.601929,29697
"""2019-07-04""",16,276.918304,-1.868439,14
"""2019-07-03""",6,280.61853,-1.983429,58019


In [ ]:
display(suhi_lcz.fig)

**Interpretando.** As barras são ΔT relativo à LCZ 14 (vegetação rasteira), uma barra por classe com barra de erro (IC 95% entre as datas). Classes compactas, verticalizadas ou fortemente pavimentadas (LCZ 1–3, LCZ 8 baixa-densidade grande) tipicamente mostram o maior ΔT positivo; classes abertas/vegetadas (LCZ 9, LCZ A–G) ficam perto de zero ou negativas. Este é o diagnóstico a usar quando se quer identificar *qual forma construída* mais impulsiona a SUHI, por exemplo para priorizar onde telhados frios ou arborização ajudariam mais.

### 4c — `method="utfvi"`: Índice de Variância do Campo Térmico Urbano

O UTFVI (Liu & Zhang, 2011) é `(LST − LST_média) / LST_média`, calculado em Kelvin (uma razão de temperatura absoluta — a função converte internamente entradas em Celsius), para uma única data (`date_index`, padrão a mais recente). É então classificado em seis categorias ecológicas padrão de "excelente" a "péssimo". Diferente dos dois métodos acima, este é um **mapa espacial**, não uma série temporal — responde "onde, nesse dia específico, o estresse térmico é pior", em um índice normalizado comparável entre cidades e datas porque é relativo à própria média da cena naquela data, não a uma temperatura absoluta.

In [10]:
suhi_utfvi = lcz_uhi_surface(map_path, lst, method="utfvi", lang="pt")
suhi_utfvi.df

category,count,percent
str,u64,f64
"""bad""",15667,10.719515
"""excellent""",67784,46.378477
"""good""",27458,18.787033
"""normal""",23496,16.076194
"""worse""",7966,5.450415
"""worst""",3783,2.588366


In [ ]:
display(suhi_utfvi.fig["map"])

In [ ]:
display(suhi_utfvi.fig["bar"])

**Interpretando.** O mapa mostra o valor de UTFVI por pixel (escala divergente, centrada em zero); a barra mostra a fração de pixels válidos em cada uma das seis categorias (excelente → péssimo). Uma cidade dominada por pixels "ruim"/"pior"/"péssimo" tem estresse térmico generalizado em relação ao seu próprio contexto regional naquele dia, não apenas alguns pontos isolados — um complemento útil ao diagnóstico pontual de hotspot abaixo.

### 4d — `method="hotspot"`: detecção de hotspots espaciais Getis-Ord Gi\*

Aplica a estatística clássica de Getis-Ord Gi\* (Ord & Getis, 1995) diretamente sobre o raster de LST via uma janela móvel local (`hotspot_window`, padrão 3×3 — sem necessidade de matriz de pesos espaciais vetorial), classificando cada pixel como **Hot spot**, **Cold spot** ou **Not significant** estatisticamente significativo no nível de confiança bicaudal escolhido (`hotspot_confidence`, padrão 0.95, ou seja |z| ≥ 1.96). Diferente do UTFVI (que classifica por desvio absoluto da média), o Gi\* pergunta se o valor de um pixel *junto com o de seus vizinhos* é significativamente maior/menor do que poderia surgir ao acaso — é um verdadeiro teste de agrupamento espacial, não um limiar por pixel, filtrando pixels ruidosos isolados e destacando clusters coerentes de calor/frio.

In [13]:
suhi_hot = lcz_uhi_surface(
    map_path, lst, method="hotspot",
    hotspot_window=3, hotspot_confidence=0.95, lang="pt",
)
suhi_hot.df

category,count,percent
str,u64,f64
"""Cold spot""",5750,3.93426
"""Hot spot""",1765,1.207647
"""Not significant""",138637,94.858093


In [ ]:
display(suhi_hot.fig)

**Interpretando.** Pixels rotulados "Hot spot" são locais onde a superfície é de fato anômala e *significativamente* quente como parte de um cluster espacialmente coerente (ex.: um distrito industrial/comercial contíguo), o que é uma afirmação mais forte do que "este pixel isolado tem LST alta". Este é o método a usar quando se precisa de fronteiras de hotspot estatisticamente defensáveis — por exemplo, para priorizar bairros específicos para uma intervenção de resfriamento, em vez de reagir a pixels isolados ruidosos.

*(Não demonstrados aqui, mas disponíveis: `method="percentile"` — como `urban_rural`, mas com uma linha de base rural de percentil baixo em vez da média, útil como referência de fundo mais conservadora, embora a docstring do módulo alerte que é sensível a contaminação por nuvem não mascarada no caminho Sentinel-3; `method="transect"` — anomalia de LST agrupada por distância do pixel urbano mais próximo, o clássico perfil de gradiente rural-urbano; `method="persistence"` — a fração das datas observadas em que cada pixel excedeu a referência rural do dia por um limiar, respondendo "quão confiavelmente quente é este local" em vez de mostrar um instantâneo de uma única data. Execute `lcz_uhi_surface(map_path, lst, method="all")` para obter todos os métodos de uma vez como um dict.)*

## Conclusão

Você calculou a intensidade da UHI de duas formas para Berlim: a UHI de dossel a partir da temperatura do ar de estações (`lcz_uhi_intensity`, estratificação de estações LCZ 1–10 vs. 11–16), e a UHI de Superfície a partir da LST do Sentinel-3 (`lcz_uhi_surface`, quatro de seus sete métodos diagnósticos: `urban_rural`, `lcz`, `utfvi`, `hotspot`). As duas raramente concordam em magnitude ou horário diurno — isso é esperado, não um erro, já que medem grandezas físicas diferentes (ar vs. temperatura da pele) em horários diferentes do dia. A classificação urbana/rural baseada em LCZ fundamenta ambas, dando uma definição fisicamente fundamentada e independente da forma da cidade, em vez de uma divisão arbitrária centro-vs-subúrbio.

Notebook anterior: `notebooks/local/02_temperature_anomalies`. Próximo notebook: `notebooks/local/04_spatial_interpolation_geostats`, que interpola essas observações de estações em uma grade espacial contínua.

## Figuras de UHI prontas para publicação com `style`

As funções de plotagem agora aceitam `style="default"`, `style="nature"`, `style="science"` e `style="generic_bw"`. Os presets de revista ajustam a família de fontes, o tamanho físico em milímetros, o DPI (300 para os estilos voltados à publicação) e as cores, para que tanto o gráfico de UHI de dossel quanto os mapas de SUHI possam ser exportados em formato de publicação sem estilização extra.

Para os diagnósticos de SUHI baseados em mapa, também estão disponíveis os auxiliares cartográficos automáticos: `add_scalebar=True` e `add_north_arrow=True`.

Um exemplo curto pronto para publicação:

- `style="science"` para o gráfico de linha de UHI.
- `style="nature"` para o painel de mapa de SUHI.
- `isave=True` com `save_extension="png"` ou `"pdf"` para exportar a figura final.


In [ ]:
fig_uhi_pub = lcz_uhi_intensity(
    map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    time_freq="day",
    method="LCZ",
    group=True,
    iplot=True,
    style="science",
    isave=True,
    save_extension="png",
    lang="pt",
)
display(fig_uhi_pub)

suhi_ur_pub = lcz_uhi_surface(
    map_path,
    lst,
    method="urban_rural",
    style="nature",
    isave=True,
    save_extension="pdf",
    add_scalebar=True,
    add_north_arrow=True,
    lang="pt",
)
display(suhi_ur_pub.fig)
